# A Socratic Journey Through Neural Network Quantization
### *From First Principles to Pareto-Optimal Sub-3-Bit Vision Models*

Welcome to this interactive study! Rather than presenting quantization as an arbitrary set of recipes, this notebook guides you through an inquiry-driven exploration of:
1. **Act 0: Foundations** — How neural networks propagate variance, why quantization can make signals vanish to zero or explode to `inf`/`NaN`, and the clipping trade-off.
2. **Act I: The Free Lunch Illusion** — Why does Global INT8 work with virtually zero effort?
3. **Act II: The Catastrophic Collapse** — Why does Global INT4 plummet to 10% random guessing?
4. **Act III: Isolating Outliers** — Can per-channel scaling and OMSE clipping rescue 4-bit?
5. **Act IV: Sub-Channel Granularity** — How group-wise blocks and asymmetric zero-points push 3-bit.
6. **Act V: The Architectural Revelation** — Exploiting ResNet-18 parameter asymmetry with Sensitivity-Guided Pareto optimization to beat FP32 at 2.5 effective bits!

In [ ]:
import sys
from pathlib import Path
import copy
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Ensure src/ is on sys.path
NOTEBOOK_DIR = Path(".").resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(ROOT / "src"))

from models.vision import load_pretrained_model
from data.imagenette import make_test_loader, make_calibration_loader
from evaluation.metrics import evaluate
from quantization.integer.weight import (
    quantize_weights,
    quantize_per_channel,
    quantize_groupwise,
    compute_layer_sensitivities,
    quantize_sensitivity_guided,
)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Act 0: Neural Network Fundamentals & Quantization Mechanics

Before quantizing a real model, we must understand the physics of signal propagation through deep architectures.

### 1. How Deep Networks Propagate Signals
A deep neural network computes a cascade of $L$ layer transformations:
$$x_{l+1} = \sigma(W_l x_l + b_l)$$

Under standard initialization (He / Xavier), the weights are scaled so that the **effective layer gain** $g_l$ preserves activation variance:
$$\mathbb{E}[\|x_{l+1}\|^2] \approx g_l^2 \cdot \mathbb{E}[\|x_l\|^2] \quad \text{where } g_l \approx 1$$

Across $L = 18$ layers in ResNet-18, the signal magnitude at the final layer scales as:
$$\|x_L\| \approx \left(\prod_{l=1}^{L} g_l\right) \|x_0\| = g^L \|x_0\|$$

---

### 2. The Anatomy of Integer Quantization
Quantization maps continuous floating-point weights $w \in [\alpha, \beta]$ to a discrete $b$-bit integer grid:
$$s = \frac{\beta - \alpha}{2^b - 1}, \quad z = \text{round}\left(-\frac{\alpha}{s}\right)$$
$$q = \text{clamp}\left(\left\lfloor \frac{w}{s} \right\rceil + z, 0, 2^b - 1\right)$$
$$\hat{w} = (q - z) \cdot s$$

Here:
- $s$ (the **step size** or **scale**) is the distance between consecutive discrete levels.
- The interval $\left[-\frac{s}{2}, +\frac{s}{2}\right]$ is the **zero-bin**: any weight with magnitude $|w| < \frac{s}{2}$ rounds directly to **0.0**.

---

### 3. Why Quantization Causes Values to Approach Zero (Signal Vanishing)
- If the scale $s$ is too coarse (or inflated by an outlier), a vast percentage of weights fall inside the zero-bin.
- When 90–99% of weights in a matrix become zero, the effective layer gain drops below unity: $g \ll 1$.
- Over 18 layers, this compounding attenuation causes the signal to vanish:
$$g = 0.6 \implies g^{18} = (0.6)^{18} \approx 0.0001$$
- Combined with non-linearities like $\text{ReLU}(x) = \max(0, x)$, when activations drop below zero, neurons permanently "die", and the network produces static or random predictions (10.0% accuracy on 10 classes).

---

### 4. Why Quantization / Clipping Causes Values to Blow Up (`inf` & `NaN`)
- Conversely, suppose the quantization scale is mismatched with true weight magnitudes.
- In naive 1-bit quantization, binarizing BatchNorm scales $\gamma$ (max $\approx 2.40$) applied a model-wide scale $s = 2.4075$ to deep layers where true weight magnitudes averaged only $0.015$.
- This imposed an artificial per-layer gain of:
$$g = \frac{2.4075}{0.015} \approx 160\times$$
- Over 18 layers, signal magnitude compounded exponentially:
$$\|x_{18}\| \approx 160^{18} \approx 1.2 \times 10^{39}$$
- In IEEE-754 Single-Precision Floating Point (`float32`), the maximum representable finite number is:
$$\text{float32}_{\max} = (2 - 2^{-23}) \times 2^{127} \approx 3.4028 \times 10^{38}$$
- Because $1.2 \times 10^{39} > 3.4 \times 10^{38}$, the numbers **overflow directly to `+inf`**!
- In subsequent layers, operations like $\infty - \infty$ or $\text{Softmax}(\infty)$ yield **`NaN`** (Not-a-Number), crashing the model completely.

---

### 5. The Weight Clipping Dilemma
When quantizing to low bit-widths (e.g., 4-bit), we face an inherent mathematical tension:
$$\text{Total Error} = \text{Clipping Error} + \text{Rounding Error}$$
- **No clipping ($\alpha = \max|W|$):** Step size $s = \frac{2\alpha}{2^b - 1}$ is stretched by extreme outliers. Rounding noise dominates, drowning normal weights into zero.
- **Aggressive clipping ($\alpha \ll \max|W|$):** Step size $s$ shrinks, giving high resolution to the dense core, but outlier weights are heavily truncated.
- **Optimal MSE Clipping (OMSE):** Finds the sweet spot $\alpha^* \in [0.6, 1.0]$ that balances clipping noise and rounding noise to minimize total L2 error!

In [ ]:
# Interactive Simulation: Signal Propagation Across 18 Layers
# Watch signals stay balanced, vanish to zero, or blow up exponentially!

def simulate_propagation():
    L = 18
    dim = 256
    torch.manual_seed(42)
    x0 = torch.randn(1, dim)

    # Condition 1: Balanced FP32 (gain g ~ 1.0)
    norm_fp32 = [x0.norm().item()]
    x = x0.clone()
    for _ in range(L):
        w = torch.randn(dim, dim) / (dim ** 0.5)
        x = torch.relu(x @ w) * (2 ** 0.5)
        norm_fp32.append(x.norm().item())

    # Condition 2: Coarse Quantization / Over-Pruned (75% weights zeroed -> g ~ 0.55)
    norm_zero = [x0.norm().item()]
    x = x0.clone()
    for _ in range(L):
        w = torch.randn(dim, dim) / (dim ** 0.5)
        mask = (torch.rand_like(w) > 0.75).float()
        w = w * mask
        x = torch.relu(x @ w) * (2 ** 0.5)
        norm_zero.append(x.norm().item())

    # Condition 3: Gain Mismatch / Amplification (gain g ~ 1.75)
    norm_blowup = [x0.norm().item()]
    x = x0.clone()
    for _ in range(L):
        w = torch.randn(dim, dim) / (dim ** 0.5) * 1.75
        x = torch.relu(x @ w) * (2 ** 0.5)
        norm_blowup.append(x.norm().item())

    plt.figure(figsize=(10, 4.5))
    plt.plot(range(L + 1), norm_fp32, label="Balanced FP32 (Stable Variance)", color="#2ecc71", linewidth=2.5)
    plt.plot(range(L + 1), norm_zero, label="Signal Vanishing (Weights drowned in zero-bin)", color="#3498db", linewidth=2.5)
    plt.plot(range(L + 1), norm_blowup, label="Signal Explosion (Mismatched scale amplification)", color="#e74c3c", linewidth=2.5)

    plt.yscale("log")
    plt.xlabel("Layer Depth ($l = 1 \dots 18$)", fontsize=11)
    plt.ylabel("Activation Vector Norm $\|x_l\|$ (Log Scale)", fontsize=11)
    plt.title("The Physics of Deep Signal Propagation: Why Quantization Stability Matters", fontsize=12, fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

simulate_propagation()

In [ ]:
torch.set_num_threads(4)
device = torch.device("cpu")
checkpoint_path = ROOT / "results/checkpoints/resnet18_pretrained_fp32.pt"

# Load torchvision pretrained ResNet-18
model, weights = load_pretrained_model("resnet18", checkpoint_path)
model = model.to(device).eval()

# Load 500-sample Imagenette test split and calibration split
test_loader, class_indices = make_test_loader(
    ROOT / "data/raw/imagenette2-160",
    weights,
    batch_size=32,
    images_per_class=50,
    seed=42,
)

calibration_loader = make_calibration_loader(
    ROOT / "data/raw/imagenette2-160",
    weights,
    batch_size=32,
    max_images=128,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded Pretrained ResNet-18: {total_params:,} parameters ({total_params*4/1e6:.2f} MB in FP32)")
print(f"Validation Dataset: Imagenette ({len(test_loader.dataset)} images across 10 classes)")

## Act I: The Illusion of the Free Lunch
> **The Socratic Question:** *Can we simply round 32-bit floating point weights into 8-bit integers?*

Let's test the baseline FP32 model and apply a single, model-wide symmetric scale factor:
$$s = \frac{\max(|W|)}{2^{b-1} - 1}$$
$$q = \text{clamp}\left(\text{round}\left(\frac{W}{s}\right), -128, 127\right), \quad \hat{W} = q \cdot s$$

Let's evaluate FP32 against Global INT8.

In [ ]:
# 1. Baseline FP32
fp32_res = evaluate(model, test_loader, device, class_indices)
print(f"Baseline FP32 Accuracy: {fp32_res['accuracy']*100:.2f}% | Loss: {fp32_res['loss']:.4f}")

# 2. Global INT8
m_int8 = copy.deepcopy(model)
quantize_weights(m_int8, bits=8)
int8_res = evaluate(m_int8, test_loader, device, class_indices)
print(f"Global INT8 Accuracy:   {int8_res['accuracy']*100:.2f}% | Loss: {int8_res['loss']:.4f}")
print(f"Storage reduction:      46.76 MB -> 11.72 MB (4x compression!)")

## Act II: The Catastrophic Collapse
> **The Socratic Question:** *If 8-bit was so effortless, why does Global INT4 collapse to 10% random guessing?*

Let's quantize to Global INT4 ($b=4$) and inspect what happened mathematically to the weights!

In [ ]:
m_int4 = copy.deepcopy(model)
meta_int4 = quantize_weights(m_int4, bits=4)
int4_res = evaluate(m_int4, test_loader, device, class_indices)
print(f"Global INT4 Accuracy: {int4_res['accuracy']*100:.2f}% (Complete Collapse!)")

# Inspect weight distributions:
conv1_w = dict(model.named_modules())["conv1"].weight.data
l4_w = dict(model.named_modules())["layer4.1.conv2"].weight.data

scale = meta_int4["scale"]
zero_thresh = scale / 2.0

print(f"\nDiagnostic Analysis:")
print(f"-> Global Scale (set by conv1 max): {scale:.4f}")
print(f"-> Zero-bin threshold [-thresh, +thresh]: [-{zero_thresh:.4f}, +{zero_thresh:.4f}]")
print(f"-> conv1 max magnitude:                  {conv1_w.abs().max():.4f}")
print(f"-> layer4.1.conv2 mean magnitude:        {l4_w.abs().mean():.4f}")

In [ ]:
# Measure percentage of weights turned to EXACT ZERO in each layer under Global INT4
layer_names = []
zero_percentages = []

for name, module in m_int4.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        w_q = module.weight.data
        zero_pct = (w_q == 0.0).float().mean().item() * 100.0
        layer_names.append(name.replace("layer", "L").replace(".conv", ".c"))
        zero_percentages.append(zero_pct)

plt.figure(figsize=(12, 4))
plt.bar(range(len(layer_names)), zero_percentages, color="#e74c3c", edgecolor="black")
plt.xticks(range(len(layer_names)), layer_names, rotation=60, ha="right", fontsize=9)
plt.ylabel("% Weights Mapped to 0.0", fontsize=11)
plt.title("The Outlier Tyranny: 99.9% of Deep Layer Weights Rounded to 0.0 under Global INT4", fontsize=12, fontweight="bold")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Act III: Isolating Outliers (Per-Channel & OMSE Clipping)
> **The Socratic Question:** *Can we prevent one outlier channel from corrupting the rest? And can deliberately throwing away outlier tails IMPROVE accuracy?*

1. **Per-Channel Quantization:** Assign one scale per output filter (dim 0).
2. **Optimal MSE (OMSE) Clipping (Banner et al., NeurIPS 2019):**
   By shrinking $\alpha \in [0.6, 1.0]$, we clip the outer 10% tails, cutting $\Delta$ in half for the dense 90% body!

In [ ]:
# 1. Per-Channel INT4 Standard
m_pc = copy.deepcopy(model)
quantize_per_channel(m_pc, bits=4, mse_clipping=False)
pc_res = evaluate(m_pc, test_loader, device, class_indices)
print(f"Per-Channel INT4 (PyTorch Standard): {pc_res['accuracy']*100:.2f}% (Jump from 10.0% -> 89.6%!)")

# 2. Per-Channel INT4 + OMSE Clipping
m_omse = copy.deepcopy(model)
quantize_per_channel(m_omse, bits=4, mse_clipping=True)
omse_res = evaluate(m_omse, test_loader, device, class_indices)
print(f"Per-Channel INT4 + OMSE Clipping:    {omse_res['accuracy']*100:.2f}% (Gained another +3.4%!)")

## Act IV: Sub-Channel Granularity (Group-wise & Asymmetric Quantization)
> **The Socratic Question:** *In deep layers, a single channel has 4,608 weights. Can we go more granular than a whole channel?*

1. **Group-wise / Block-wise ($G=32$):** Divides weights into contiguous blocks of 32 elements (AWQ / GPTQ / GGUF style).
2. **Asymmetric vs Symmetric:** Asymmetric adds a zero-point $z = \text{round}(-\min / s)$, utilizing all $2^b$ levels even when weights are skewed!

In [ ]:
# Group-wise INT4 (G=32)
m_gw4_sym = copy.deepcopy(model)
quantize_groupwise(m_gw4_sym, bits=4, group_size=32, asymmetric=False)
gw4_sym = evaluate(m_gw4_sym, test_loader, device, class_indices)

m_gw4_asym = copy.deepcopy(model)
quantize_groupwise(m_gw4_asym, bits=4, group_size=32, asymmetric=True)
gw4_asym = evaluate(m_gw4_asym, test_loader, device, class_indices)

# Group-wise INT3 (G=32)
m_gw3_sym = copy.deepcopy(model)
quantize_groupwise(m_gw3_sym, bits=3, group_size=32, asymmetric=False)
gw3_sym = evaluate(m_gw3_sym, test_loader, device, class_indices)

m_gw3_asym = copy.deepcopy(model)
quantize_groupwise(m_gw3_asym, bits=3, group_size=32, asymmetric=True)
gw3_asym = evaluate(m_gw3_asym, test_loader, device, class_indices)

print(f"Group-wise INT4 (G=32, Symmetric):  {gw4_sym['accuracy']*100:.2f}%")
print(f"Group-wise INT4 (G=32, Asymmetric): {gw4_asym['accuracy']*100:.2f}% (Only 0.8% drop from FP32!)")
print(f"Group-wise INT3 (G=32, Symmetric):  {gw3_sym['accuracy']*100:.2f}%")
print(f"Group-wise INT3 (G=32, Asymmetric): {gw3_asym['accuracy']*100:.2f}% (+9.2% jump from zero-points!)")

## Act V: Breaking Uniform Symmetry (Sensitivity-Guided Mixed Precision)
> **The Socratic Question:** *Must every layer have the exact same bit-width? Can we reach sub-3-bit and 2-bit with >95% accuracy?*

Let's inspect the parameter distribution of ResNet-18:

In [ ]:
modules = [(name, m) for name, m in model.named_modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
total_w = sum(m.weight.numel() for _, m in modules)

counts = {
    "conv1 & layer1": sum(m.weight.numel() for name, m in modules if "conv1" in name or "layer1" in name),
    "layer2": sum(m.weight.numel() for name, m in modules if "layer2" in name),
    "layer3": sum(m.weight.numel() for name, m in modules if "layer3" in name),
    "layer4 (Bulk)": sum(m.weight.numel() for name, m in modules if "layer4" in name),
    "fc (Classifier)": sum(m.weight.numel() for name, m in modules if "fc" in name),
}

plt.figure(figsize=(7, 7))
plt.pie(
    counts.values(),
    labels=[f"{k}\n({v:,} | {v/total_w*100:.1f}%)" for k, v in counts.items()],
    autopct="%1.1f%%",
    startangle=140,
    colors=["#3498db", "#2ecc71", "#f39c12", "#e74c3c", "#9b59b6"],
    explode=[0.05, 0.05, 0.05, 0.08, 0.05],
)
plt.title("ResNet-18 Parameter Distribution: layer4 Contains 75.1% of All Weights!", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
print("Computing layer-wise output perturbation sensitivities on calibration data...")
sens = compute_layer_sensitivities(
    model, calibration_loader, device,
    candidate_bits=(1, 2, 3, 4, 8),
    group_size=32,
    asymmetric=True,
    max_images=128,
)

budgets = [2.0, 2.2, 2.5, 3.0, 3.5, 4.0]
pareto_results = []

for b in budgets:
    m_sg = copy.deepcopy(model)
    meta = quantize_sensitivity_guided(
        m_sg, target_effective_bits=b, sensitivities=sens, group_size=32, asymmetric=True
    )
    res = evaluate(m_sg, test_loader, device, class_indices)
    actual_b = meta["effective_weight_bits"]
    pareto_results.append((actual_b, res["accuracy"] * 100))
    print(f"Target Budget {b:.1f}b -> Actual: {actual_b:.2f}b | Accuracy: {res['accuracy']*100:.2f}% | Loss: {res['loss']:.4f}")

In [ ]:
bits_vals, acc_vals = zip(*pareto_results)

plt.figure(figsize=(10, 5))
plt.axhline(fp32_res["accuracy"] * 100, color="gray", linestyle="--", label=f"FP32 Baseline ({fp32_res['accuracy']*100:.1f}%)")

# Baselines for comparison
plt.scatter([1.0, 2.0, 4.0], [10.0, 10.0, 10.0], color="red", marker="x", s=80, label="Blind Global (INT1, INT2, INT4)")
plt.scatter([4.0], [pc_res["accuracy"]*100], color="orange", marker="^", s=80, label="Per-Channel INT4 (89.6%)")
plt.scatter([4.0], [gw4_asym["accuracy"]*100], color="purple", marker="s", s=80, label="Group-wise INT4 (96.6%)")
plt.scatter([3.0], [gw3_asym["accuracy"]*100], color="brown", marker="d", s=80, label="Group-wise INT3 (91.8%)")

# Pareto curve
plt.plot(bits_vals, acc_vals, color="#27ae60", marker="o", linewidth=2.5, markersize=8, label="Sensitivity-Guided Pareto Frontier")

for b, a in zip(bits_vals, acc_vals):
    plt.annotate(f"{a:.1f}% ({b:.2f}b)", (b, a), textcoords="offset points", xytext=(-10, 8), fontweight="bold", color="#196f3d")

plt.xlabel("Effective Weight Bits", fontsize=11)
plt.ylabel("Top-1 Accuracy (%)", fontsize=11)
plt.title("The Quantization Pareto Frontier: Maximizing Accuracy with Minimal Effective Bits", fontsize=12, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Interactive Playground: Custom Bit Budget Allocator
Try entering any custom target bit budget (e.g., `2.3`, `2.7`, or `3.2`) to see how the solver dynamically allocates bits across layers to maximize accuracy!

In [ ]:
def test_custom_budget(target_bits=2.5):
    m_custom = copy.deepcopy(model)
    meta = quantize_sensitivity_guided(
        m_custom, target_effective_bits=target_bits, sensitivities=sens, group_size=32, asymmetric=True
    )
    res = evaluate(m_custom, test_loader, device, class_indices)
    print(f"=== Budget: {target_bits:.2f} bits ===")
    print(f"Achieved Effective Bits: {meta['effective_weight_bits']:.2f}")
    print(f"Accuracy:                {res['accuracy']*100:.2f}% (FP32 baseline is {fp32_res['accuracy']*100:.2f}%)")
    print(f"Estimated Storage:       {meta['scale_count']*5/1e6 + meta['quantized_weight_count']*meta['effective_weight_bits']/8/1e6:.2f} MB")
    print("\nLayer Precision Allocation:")
    for name, d in meta["layers"].items():
        print(f"  {name:30s} -> {d['bits']} bits ({d['weights']:,} weights)")

# Try a budget of 2.3 bits:
test_custom_budget(2.3)